# Neural Field Diffusion on ShapeNetCore.v2

This notebook trains the neural field diffusion model on real 3D shapes from ShapeNetCore.v2.

**Dataset**: [ShapeNetCore.v2](https://www.kaggle.com/datasets/hajareddagni/shapenetcorev2)
- ~51,300 unique 3D models
- 55 common object categories
- OBJ mesh format

**Key Differences from Toy Data**:
1. Real complex geometry (not parametric)
2. Large variety within categories
3. Need to sample point clouds from meshes
4. Requires larger model capacity

## Setup

1. Download ShapeNetCore.v2 from Kaggle
2. Extract to `data/shapenet/` directory
3. Run this notebook

In [ ]:
import sys
sys.path.append('..')

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm
from pathlib import Path
import time
import trimesh
import warnings
warnings.filterwarnings('ignore')

# Our modules
from src.models.neural_field import NeuralFieldDiffusion
from src.models.sdf_field import SDFNeuralField, SDFFlowMatchingLoss
from src.diffusion.flow_matching import FlowMatchingLoss, FlowMatchingSampler

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Check for trimesh
try:
    import trimesh
    print(f"trimesh version: {trimesh.__version__}")
except ImportError:
    print("Please install trimesh: pip install trimesh")
    print("For faster mesh loading: pip install trimesh[easy]")

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Data paths - UPDATE THIS TO YOUR SHAPENET LOCATION
# Common locations to check:
SHAPENET_PATHS = [
    Path('../data/shapenet/ShapeNetCore.v2'),
    Path('../data/ShapeNetCore.v2'),
    Path.home() / '.cache/kagglehub/datasets/hajareddagni/shapenetcorev2/versions/1/ShapeNetCore.v2/ShapeNetCore.v2',
    Path('/home/idies/workspace/Temporary/dpark1/scratch/conda/conda_envs/mamba/.cache/kagglehub/datasets/hajareddagni/shapenetcorev2/versions/1/ShapeNetCore.v2/ShapeNetCore.v2'),
]

# Auto-detect ShapeNet location
SHAPENET_ROOT = None
for path in SHAPENET_PATHS:
    if path.exists():
        # Check if it contains synset folders (like 02691156)
        synset_dirs = [d for d in path.iterdir() if d.is_dir() and d.name.isdigit()]
        if synset_dirs:
            SHAPENET_ROOT = path
            break

# If not found, use first path as default (user will need to update)
if SHAPENET_ROOT is None:
    SHAPENET_ROOT = SHAPENET_PATHS[0]
    print(f"WARNING: ShapeNet not auto-detected!")
    print(f"Please set SHAPENET_ROOT to your ShapeNet location")
    print(f"Expected folder structure: SHAPENET_ROOT/02691156/... (synset folders)")

# ShapeNet category IDs (synset IDs)
# See full list: https://shapenet.org/
CATEGORY_IDS = {
    'airplane': '02691156',
    'car': '02958343',
    'chair': '03001627',
    'table': '04379243',
    'sofa': '04256520',
    'lamp': '03636649',
    'vessel': '04530566',
    'rifle': '04090263',
    'speaker': '03691459',
    'bench': '02828884',
    'cabinet': '02933112',
    'display': '03211117',
    'phone': '04401088',
    'watercraft': '04530566',
}

# Categories to train on (start with one for testing)
TRAIN_CATEGORIES = ['chair']  # Try: ['airplane'], ['car'], or multiple

# Data settings
N_POINTS = 2048          # Points per shape (more for real data)
MAX_SHAPES_PER_CAT = 500 # Limit for faster testing (None for all)
NORMALIZE = True
AUGMENT = True           # Random rotation during training

# Model (LARGER for real data)
MODEL_TYPE = 'velocity'  # 'velocity' or 'sdf'
HIDDEN_SIZE = 384        # Larger than toy
HIDDEN_SIZE_X = 64
NUM_HEADS = 6
NUM_BLOCKS = 12
NUM_COND_BLOCKS = 4
NERF_MLP_RATIO = 4
MAX_FREQS = 8

# Training
EPOCHS = 200
BATCH_SIZE = 8           # Smaller due to more points
LR = 1e-4
GRADIENT_ACCUMULATION = 4  # Effective batch = 32

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Categories: {TRAIN_CATEGORIES}")
print(f"Model type: {MODEL_TYPE}")
print(f"ShapeNet path: {SHAPENET_ROOT}")

# Verify data path and show available categories
if SHAPENET_ROOT.exists():
    synset_dirs = sorted([d.name for d in SHAPENET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
    print(f"✓ ShapeNet found with {len(synset_dirs)} categories")
    # Show which requested categories exist
    for cat in TRAIN_CATEGORIES:
        synset = CATEGORY_IDS.get(cat, cat)
        if synset in synset_dirs:
            cat_path = SHAPENET_ROOT / synset
            n_shapes = len([d for d in cat_path.iterdir() if d.is_dir()])
            print(f"  ✓ {cat} ({synset}): {n_shapes} shapes")
        else:
            print(f"  ✗ {cat} ({synset}): NOT FOUND")
else:
    print(f"✗ ShapeNet NOT found at: {SHAPENET_ROOT}")

## 2. ShapeNet Dataset

In [ ]:
class ShapeNetDataset(Dataset):
    """
    ShapeNetCore.v2 Dataset for point cloud generation.
    
    Loads meshes (OBJ or PLY) and samples point clouds from surfaces.
    """
    
    def __init__(self, root_dir, categories, n_points=2048, 
                 max_shapes_per_cat=None, normalize=True, augment=True,
                 cache_dir=None, debug=False):
        """
        Args:
            root_dir: Path to ShapeNetCore.v2
            categories: List of category names or synset IDs
            n_points: Number of points to sample per shape
            max_shapes_per_cat: Limit shapes per category (for testing)
            normalize: Whether to normalize to unit sphere
            augment: Whether to apply random rotation
            cache_dir: Directory to cache sampled point clouds
            debug: Print debug info about directory structure
        """
        self.root_dir = Path(root_dir)
        self.n_points = n_points
        self.normalize = normalize
        self.augment = augment
        self.cache_dir = Path(cache_dir) if cache_dir else None
        
        # Map category names to synset IDs
        self.synset_ids = []
        for cat in categories:
            if cat in CATEGORY_IDS:
                self.synset_ids.append(CATEGORY_IDS[cat])
            else:
                self.synset_ids.append(cat)  # Assume it's already a synset ID
        
        # Find all shape directories
        self.shape_paths = []
        self.shape_categories = []
        
        for synset_id in self.synset_ids:
            cat_dir = self.root_dir / synset_id
            if not cat_dir.exists():
                print(f"Warning: Category dir {cat_dir} not found")
                continue
            
            shape_dirs = sorted([d for d in cat_dir.iterdir() if d.is_dir()])
            
            if debug and shape_dirs:
                print(f"\nDebug: Category {synset_id}")
                print(f"  Found {len(shape_dirs)} shape directories")
                first_shape = shape_dirs[0]
                print(f"  First shape: {first_shape.name}")
                print(f"  Contents: {[x.name for x in first_shape.iterdir()]}")
                models_dir = first_shape / 'models'
                if models_dir.exists():
                    print(f"  models/ contents: {[x.name for x in models_dir.iterdir()]}")
                # Check for mesh files (OBJ or PLY)
                mesh_files = list(first_shape.rglob('*.obj')) + list(first_shape.rglob('*.ply'))
                print(f"  Mesh files found: {[str(f.relative_to(first_shape)) for f in mesh_files[:5]]}")
            
            if max_shapes_per_cat:
                shape_dirs = shape_dirs[:max_shapes_per_cat]
            
            for shape_dir in shape_dirs:
                model_path = None
                
                # Try common ShapeNet paths - BOTH OBJ AND PLY
                candidates = [
                    # PLY files (Kaggle version)
                    shape_dir / 'models' / 'model_normalized.ply',
                    shape_dir / 'models' / 'model.ply',
                    shape_dir / 'model_normalized.ply',
                    shape_dir / 'model.ply',
                    # OBJ files (standard version)
                    shape_dir / 'models' / 'model_normalized.obj',
                    shape_dir / 'models' / 'model.obj',
                    shape_dir / 'model_normalized.obj',
                    shape_dir / 'model.obj',
                ]
                
                for candidate in candidates:
                    if candidate.exists():
                        model_path = candidate
                        break
                
                # If not found, search for any mesh file
                if model_path is None:
                    mesh_files = list(shape_dir.rglob('*.ply')) + list(shape_dir.rglob('*.obj'))
                    if mesh_files:
                        model_path = mesh_files[0]
                
                if model_path is not None:
                    self.shape_paths.append(model_path)
                    self.shape_categories.append(synset_id)
        
        print(f"Found {len(self.shape_paths)} shapes across {len(self.synset_ids)} categories")
        
        if len(self.shape_paths) > 0:
            print(f"  Using format: {self.shape_paths[0].suffix}")
        
        # Setup cache
        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def __len__(self):
        return len(self.shape_paths)
    
    def _get_cache_path(self, idx):
        """Get cache file path for a shape."""
        if self.cache_dir is None:
            return None
        shape_id = self.shape_paths[idx].parent.parent.name
        cat_id = self.shape_categories[idx]
        return self.cache_dir / f"{cat_id}_{shape_id}_{self.n_points}.npy"
    
    def _load_and_sample(self, idx):
        """Load mesh and sample points."""
        # Check cache first
        cache_path = self._get_cache_path(idx)
        if cache_path and cache_path.exists():
            return np.load(cache_path)
        
        # Load mesh (trimesh handles both OBJ and PLY)
        mesh_path = self.shape_paths[idx]
        try:
            mesh = trimesh.load(mesh_path, force='mesh')
        except Exception as e:
            print(f"Error loading {mesh_path}: {e}")
            return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Handle scene vs mesh
        if isinstance(mesh, trimesh.Scene):
            meshes = [g for g in mesh.geometry.values() if isinstance(g, trimesh.Trimesh)]
            if meshes:
                mesh = trimesh.util.concatenate(meshes)
            else:
                return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Sample points from surface
        try:
            points, _ = trimesh.sample.sample_surface(mesh, self.n_points)
            points = points.astype(np.float32)
        except Exception as e:
            print(f"Error sampling {mesh_path}: {e}")
            return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Cache the result
        if cache_path:
            np.save(cache_path, points)
        
        return points
    
    def _normalize(self, points):
        """Normalize to unit sphere."""
        centroid = points.mean(axis=0)
        points = points - centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))
        if max_dist > 0:
            points = points / max_dist
        return points
    
    def _random_rotate(self, points):
        """Apply random SO(3) rotation."""
        random_matrix = np.random.randn(3, 3)
        q, r = np.linalg.qr(random_matrix)
        if np.linalg.det(q) < 0:
            q[:, 0] *= -1
        return (points @ q.T).astype(np.float32)
    
    def __getitem__(self, idx):
        points = self._load_and_sample(idx)
        
        if self.normalize:
            points = self._normalize(points)
        
        if self.augment:
            points = self._random_rotate(points)
        
        return torch.tensor(points, dtype=torch.float32)
    
    def get_category(self, idx):
        """Get category of shape at index."""
        return self.shape_categories[idx]

In [ ]:
# Create dataset with debug=True to see directory structure
cache_dir = Path('../data/shapenet_cache')

dataset = ShapeNetDataset(
    root_dir=SHAPENET_ROOT,
    categories=TRAIN_CATEGORIES,
    n_points=N_POINTS,
    max_shapes_per_cat=MAX_SHAPES_PER_CAT,
    normalize=NORMALIZE,
    augment=AUGMENT,
    cache_dir=cache_dir,
    debug=True  # Enable to see what's inside the directories
)

if len(dataset) > 0:
    dataloader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )
    print(f"\nDataset size: {len(dataset)}")
    print(f"Batches per epoch: {len(dataloader)}")
else:
    print("\n❌ Dataset is empty - cannot create dataloader")
    print("Please check the directory structure above and update ShapeNetDataset accordingly")

In [ ]:
# Visualize some training samples
fig = plt.figure(figsize=(16, 8))

for i in range(8):
    sample = dataset[i].numpy()
    
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    
    # Subsample for visualization
    vis_idx = np.random.choice(len(sample), min(1000, len(sample)), replace=False)
    vis_points = sample[vis_idx]
    
    colors = vis_points[:, 2]  # Color by z
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'Shape {i+1}')
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])
    ax.view_init(elev=20, azim=45 + i*15)

plt.suptitle(f'ShapeNet {TRAIN_CATEGORIES} - Training Samples ({N_POINTS} points)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model

In [ ]:
# Create model
if MODEL_TYPE == 'sdf':
    model = SDFNeuralField(
        in_channels=3,
        hidden_size=HIDDEN_SIZE,
        hidden_size_x=HIDDEN_SIZE_X,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        num_cond_blocks=NUM_COND_BLOCKS,
        nerf_mlp_ratio=NERF_MLP_RATIO,
        max_freqs=MAX_FREQS,
    ).to(DEVICE)
    loss_fn = SDFFlowMatchingLoss(schedule_type='linear', eikonal_weight=0.0)
else:
    model = NeuralFieldDiffusion(
        in_channels=3,
        out_channels=3,
        hidden_size=HIDDEN_SIZE,
        hidden_size_x=HIDDEN_SIZE_X,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        num_cond_blocks=NUM_COND_BLOCKS,
        nerf_mlp_ratio=NERF_MLP_RATIO,
        max_freqs=MAX_FREQS,
    ).to(DEVICE)
    loss_fn = FlowMatchingLoss(schedule_type='linear')

sampler = FlowMatchingSampler(model)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model type: {MODEL_TYPE}")
print(f"Parameters: {n_params:,}")
print(f"Architecture: {NUM_COND_BLOCKS} DiT + {NUM_BLOCKS - NUM_COND_BLOCKS} NerfBlocks")

# Test forward pass
with torch.no_grad():
    test_input = torch.randn(2, N_POINTS, 3, device=DEVICE)
    test_t = torch.rand(2, device=DEVICE)
    test_output = model(test_input, test_t)
    print(f"\nTest - Input: {test_input.shape}, Output: {test_output.shape}")

## 4. Training

In [ ]:
# Setup optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Mixed precision training (if available)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_amp else None

# Training history
train_losses = []
epoch_times = []

print(f"Optimizer: AdamW (lr={LR})")
print(f"Scheduler: CosineAnnealing")
print(f"Mixed precision: {use_amp}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION}")

In [ ]:
def train_epoch(model, dataloader, optimizer, loss_fn, device, 
                scaler=None, grad_accum=1):
    """Train for one epoch with gradient accumulation."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(dataloader):
        x0 = batch.to(device)
        
        # Forward pass with mixed precision
        if scaler is not None:
            with torch.cuda.amp.autocast():
                output = loss_fn(model, x0)
                loss = output['loss'] / grad_accum
            scaler.scale(loss).backward()
        else:
            output = loss_fn(model, x0)
            loss = output['loss'] / grad_accum
            loss.backward()
        
        total_loss += output['loss'].item()
        n_batches += 1
        
        # Gradient accumulation step
        if (batch_idx + 1) % grad_accum == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()
    
    return total_loss / n_batches


def generate_samples(model, sampler, n_samples=4, n_points=2048, n_steps=50,
                     method='sde', noise_scale=0.1, device='cpu'):
    """Generate samples."""
    model.eval()
    noise = torch.randn(n_samples, n_points, 3, device=device)
    
    with torch.no_grad():
        if method == 'euler':
            samples = sampler.sample_euler(noise, n_steps=n_steps)
        elif method == 'sde':
            samples = sampler.sample_sde(noise, n_steps=n_steps,
                                          noise_scale=noise_scale, decay='linear')
        else:
            samples = sampler.sample_euler(noise, n_steps=n_steps)
    
    return samples

In [ ]:
# Training loop
print(f"Training for {EPOCHS} epochs on {TRAIN_CATEGORIES}...")
print("="*60)

best_loss = float('inf')
pbar = tqdm(range(EPOCHS), desc="Training")

for epoch in pbar:
    start_time = time.time()
    
    loss = train_epoch(
        model, dataloader, optimizer, loss_fn, DEVICE,
        scaler=scaler, grad_accum=GRADIENT_ACCUMULATION
    )
    
    scheduler.step()
    epoch_time = time.time() - start_time
    
    train_losses.append(loss)
    epoch_times.append(epoch_time)
    
    current_lr = scheduler.get_last_lr()[0]
    pbar.set_postfix({'loss': f'{loss:.4f}', 'lr': f'{current_lr:.2e}'})
    
    # Logging
    if (epoch + 1) % 20 == 0:
        tqdm.write(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")
    
    # Save best model
    if loss < best_loss:
        best_loss = loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }, f'../experiments/outputs/shapenet_{"_".join(TRAIN_CATEGORIES)}_best.pt')

print("\n" + "="*60)
print(f"Training complete!")
print(f"Best loss: {best_loss:.4f}")
print(f"Average epoch time: {np.mean(epoch_times):.1f}s")

In [ ]:
# Plot training curve
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
if len(train_losses) > 10:
    plt.plot(train_losses[10:])
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss (after warmup)')
    plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Generate Samples

In [ ]:
# Generate samples
print("Generating samples...")

samples = generate_samples(
    model, sampler, 
    n_samples=8, 
    n_points=N_POINTS,
    n_steps=100,  # More steps for better quality
    method='sde',
    noise_scale=0.1,
    device=DEVICE
)
samples = samples.cpu().numpy()
print(f"Generated {samples.shape[0]} samples")

In [ ]:
# Compare GT vs Generated
fig = plt.figure(figsize=(16, 8))

# Ground truth
for i in range(4):
    gt = dataset[i].numpy()
    vis_idx = np.random.choice(len(gt), min(1000, len(gt)), replace=False)
    vis_points = gt[vis_idx]
    
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'GT {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

# Generated
for i in range(4):
    gen = samples[i]
    vis_idx = np.random.choice(len(gen), min(1000, len(gen)), replace=False)
    vis_points = gen[vis_idx]
    
    ax = fig.add_subplot(2, 4, i + 5, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='plasma', s=1, alpha=0.6)
    ax.set_title(f'Generated {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle(f'ShapeNet {TRAIN_CATEGORIES}: GT (top) vs Generated (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Resolution Independence

In [ ]:
# Test resolution independence
print("Testing resolution independence...")

resolutions = [512, 1024, 2048, 4096, 8192]

fig = plt.figure(figsize=(20, 4))

for i, n_pts in enumerate(resolutions):
    noise = torch.randn(1, n_pts, 3, device=DEVICE)
    
    model.eval()
    with torch.no_grad():
        sample = sampler.sample_sde(noise, n_steps=100, noise_scale=0.1, decay='linear')
    
    sample = sample.cpu().numpy()[0]
    
    # Subsample for visualization
    vis_idx = np.random.choice(len(sample), min(1000, len(sample)), replace=False)
    vis_points = sample[vis_idx]
    
    ax = fig.add_subplot(1, 5, i + 1, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'N = {n_pts}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    
    print(f"  Generated {n_pts:5d} points")

plt.suptitle('Resolution Independence on ShapeNet', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
# Save final checkpoint
categories_str = '_'.join(TRAIN_CATEGORIES)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'model_type': MODEL_TYPE,
        'hidden_size': HIDDEN_SIZE,
        'hidden_size_x': HIDDEN_SIZE_X,
        'num_heads': NUM_HEADS,
        'num_blocks': NUM_BLOCKS,
        'num_cond_blocks': NUM_COND_BLOCKS,
        'nerf_mlp_ratio': NERF_MLP_RATIO,
        'max_freqs': MAX_FREQS,
        'n_points': N_POINTS,
    },
    'train_losses': train_losses,
    'categories': TRAIN_CATEGORIES,
    'epochs': EPOCHS,
}

save_path = f'../experiments/outputs/shapenet_{categories_str}_final.pt'
torch.save(checkpoint, save_path)
print(f"Saved checkpoint to {save_path}")

## 8. Summary

### Training on ShapeNet

**Key Differences from Toy Data**:
1. **More points**: 2048+ per shape (vs 256-512)
2. **Larger model**: ~10M params (vs ~300K)
3. **Gradient accumulation**: Effective batch size matters
4. **Mixed precision**: Essential for memory efficiency
5. **Longer training**: 200+ epochs

**Recommendations**:
- Start with single category (e.g., 'chair')
- Use caching for faster data loading
- Monitor loss carefully - should decrease smoothly
- Use SDE sampling for better quality

**Next Steps**:
1. Train on multiple categories
2. Add conditional generation (category labels)
3. Evaluate with metrics (CD, EMD)
4. Compare velocity vs SDF models